In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.stats import gaussian_kde
from matplotlib.ticker import MaxNLocator

In [ ]:
df = pd.read_csv("epistasis_results_25.csv")
df = df.dropna()

In [ ]:
df.head()

In [ ]:
def regular_bin(x, y, nbins=10, method="mean", error="sem"):
    """
    Bins data using regular (linearly spaced) bins.

    Args:
        x (array-like): Independent variable.
        y (array-like): Dependent variable to be binned.
        nbins (int): Number of linearly spaced bins.
        method (str): Aggregation method ('mean', 'median', 'sum').
        error (str): 'std' for standard deviation or 'sem' for standard error of the mean.

    Returns:
        bin_centers (numpy array): Centers of the bins.
        y_binned (numpy array): Aggregated y-values.
        y_err (numpy array): Error bars (std or sem).
    """
    bins = np.linspace(min(x), max(x), nbins + 1)
    bin_indices = np.digitize(x, bins) - 1
    bin_centers = (bins[:-1] + bins[1:]) / 2
    y_binned = np.full(nbins, np.nan)
    y_err = np.full(nbins, np.nan)

    for i in range(nbins):
        mask = bin_indices == i
        if np.any(mask):
            y_vals = y[mask]
            if method == "mean":
                y_binned[i] = np.mean(y_vals)
            elif method == "median":
                y_binned[i] = np.median(y_vals)
            elif method == "sum":
                y_binned[i] = np.sum(y_vals)
            if error == "std":
                y_err[i] = np.std(y_vals)
            elif error == "sem":
                y_err[i] = np.std(y_vals) / np.sqrt(len(y_vals))
    return bin_centers, y_binned, y_err


In [ ]:
for filename in df.file.tolist():
    row = df[df.file == filename]
    study = filename.split('.csv')[0]

    fit_single_path = '20_esm2_650M_unique_single_mutations_' + filename
    fit_double_path = '20_esm2_650M_' + filename
    sing_df = pd.read_csv(fit_single_path)
    doub_df = pd.read_csv(fit_double_path)
    # Extract data
    x = sing_df['llm_single_mut']
    y = sing_df['expt_single_mut']
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)

    # Bin the data
    x1, y1, yerr = regular_bin(x, y, nbins=15)

    # Define the color map using pinks and greens
    from matplotlib.colors import LinearSegmentedColormap
    pink_green_cmap = LinearSegmentedColormap.from_list("pink_green", ["mediumseagreen", "hotpink"])

    # Plot
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(x.values, y.values, c=z, cmap=pink_green_cmap, s=50, alpha=0.5)

    # Add fitted line
    a = 1
    b = row.b1.iloc[0]
    c_param = row.c1.iloc[0]
    x_line = np.linspace(min(x), max(x), 500)
    y_line = -a * np.log(1 + np.exp(-b * (x_line + c_param)))
    ax.plot(x_line, y_line, color='black', linewidth=2, label=r'fit')
    # Add binned line
    ax.plot(x1, y1, color='black', linewidth=2,linestyle='dashed', label=r'binned data')
    ax.errorbar(x1, y1, yerr=yerr, fmt='o', color='black', ecolor='black', elinewidth=1, capsize=0, ms=10)

    ax.xaxis.set_major_locator(MaxNLocator(nbins=7))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

    # Labels and formatting
    ax.set_title(study, fontsize=20)
    plt.xlabel("$log(f_A)$", fontsize=35)
    plt.ylabel("$log(f^e_A)$", fontsize=35)
    plt.xticks(fontsize=25)
    plt.yticks(fontsize=25)
    ax.legend(fontsize=20, loc='lower right')

    plt.tight_layout()

    plt.savefig('ESM2_Figs/'+study+'_wildtype_fit2.png', dpi=300)
    plt.show()


    log_exp_double_exp_mut2_ = doub_df["Double Mutant Fitness"] - doub_df["Mut 2 fitness"]
    log_exp_double_exp_mut1_ = doub_df["Double Mutant Fitness"] - doub_df["Mut 1 fitness"]

    # Concatenate corresponding LLM values
    x = pd.concat([doub_df['mut21'], doub_df['mut12']], ignore_index=True)
    y = pd.concat([log_exp_double_exp_mut1_, log_exp_double_exp_mut2_], ignore_index=True)

    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)

    # Bin the data
    x1, y1, yerr = regular_bin(x, y, nbins=15)

    # Define the color map using pinks and greens
    from matplotlib.colors import LinearSegmentedColormap
    pink_green_cmap = LinearSegmentedColormap.from_list("pink_green", ["mediumseagreen", "hotpink"])

    # Plot
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(x.values, y.values, c=z, cmap=pink_green_cmap, s=50, alpha=0.5)

    # Add fitted line
    a = 1
    b = row.b2.iloc[0]
    c_param = row.c2.iloc[0]
    x_line = np.linspace(min(x), max(x), 500)
    y_line = -a * np.log(1 + np.exp(-b * (x_line + c_param)))
    ax.plot(x_line, y_line, color='black', linewidth=2, label=r'$mutated background$')
    ax.plot(x1, y1, color='black', linewidth=2,linestyle='dashed', label=r'binned')
    ax.errorbar(x1, y1, yerr=yerr, fmt='o', color='black', ecolor='black', elinewidth=1, capsize=0, ms=10)


    ax.xaxis.set_major_locator(MaxNLocator(nbins=7))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    # Labels and formatting
    ax.set_title(study, fontsize=20)
    plt.xlabel(r'$log(f_{A|B})$', fontsize=35)
    plt.ylabel(r'$log(f^e_{AB}/f^e_{B})$', fontsize=35)
    plt.xticks(fontsize=25)
    plt.yticks(fontsize=25)
    #ax.legend(fontsize=15)

    plt.tight_layout()

    plt.savefig('ESM2_Figs/'+study+'_background_fit2.png', dpi=300)
    plt.show()